In [9]:
import os
import pandas as pd

# Nueva ruta exacta tras la extracción
DIR_IMAGENES = r"C:\Users\ASUS\TFG_DeepSolarEye\data\Imagenes\Solar_Panel_Soiling_Image_dataset\PanelImages"

# Validación de acceso
if os.path.exists(DIR_IMAGENES):
    total = len([f for f in os.listdir(DIR_IMAGENES) if f.lower().endswith(('.jpg', '.jpeg'))])
    print(f"✅ Conexión establecida. Se han encontrado {total} imágenes.")
else:
    print("❌ La ruta no es correcta. Verifica si hay algún espacio o letra distinta.")

✅ Conexión establecida. Se han encontrado 45754 imágenes.


In [10]:
# Listamos solo archivos que terminen en .jpg o .jpeg
archivos = [f for f in os.listdir(DIR_IMAGENES) if f.lower().endswith(('.jpg', '.jpeg'))]

print(f"Número de imágenes detectadas: {len(archivos)}")

# Verificación de integridad según el dataset DeepSolarEye
if len(archivos) >= 45000:
    print("✅ El volumen de datos coincide con el dataset original.")
else:
    print(f"⚠️ Atención: Solo se detectaron {len(archivos)} imágenes. Deberían ser más de 45,000.")

Número de imágenes detectadas: 45754
✅ El volumen de datos coincide con el dataset original.


In [11]:
def build_tfg_dataframe(folder_path):
    """
    Escanea la carpeta y crea la tabla base para el análisis visual.
    """
    archivos = [f for f in os.listdir(folder_path) if f.lower().endswith(('.jpg', '.jpeg'))]
    
    df = pd.DataFrame({
        'image_id': archivos,
        'full_path': [os.path.join(folder_path, f) for f in archivos]
    })
    
    # Ordenamos por nombre para mantener la secuencia temporal (cada 5 segundos)
    df = df.sort_values('image_id').reset_index(drop=True)
    return df

df_tfg = build_tfg_dataframe(DIR_IMAGENES)
df_tfg.head()

,image_id,full_path
0,solar_Fri_Jun_16_10__0__11_2017_L_0.9061532083...,C:\Users\ASUS\TFG_DeepSolarEye\data\Imagenes\S...
1,solar_Fri_Jun_16_10__0__16_2017_L_0.9030816970...,C:\Users\ASUS\TFG_DeepSolarEye\data\Imagenes\S...
2,solar_Fri_Jun_16_10__0__1_2017_L_0.91669804403...,C:\Users\ASUS\TFG_DeepSolarEye\data\Imagenes\S...
3,solar_Fri_Jun_16_10__0__21_2017_L_0.9030816970...,C:\Users\ASUS\TFG_DeepSolarEye\data\Imagenes\S...
4,solar_Fri_Jun_16_10__0__26_2017_L_0.8960873911...,C:\Users\ASUS\TFG_DeepSolarEye\data\Imagenes\S...


In [12]:
import pandas as pd

def clean_solar_dataframe(df):
    """
    Extrae información útil del nombre largo del archivo.
    """
    # 1. Extraer la pérdida de potencia (el valor después de '_L_')
    # Según el dataset, este valor representa el impacto del soiling [cite: 105]
    df['power_loss'] = df['image_id'].str.extract(r'L_([\d.]+)').astype(float)
    
    # 2. Limpiar el nombre para ver la fecha (opcional pero útil)
    # Formato original: solar_Day_Month_Date_HH__MM__SS_YYYY...
    df['timestamp_raw'] = df['image_id'].str.split('_L_').str[0]
    
    # 3. Ordenar por nombre para asegurar la secuencia de 5 segundos [cite: 104]
    df = df.sort_values('image_id').reset_index(drop=True)
    
    return df

# Aplicamos la limpieza
df_tfg = clean_solar_dataframe(df_tfg)

# Mostramos solo las columnas importantes para confirmar
print(f"Dataset listo con {len(df_tfg)} filas.")
df_tfg[['image_id', 'power_loss']].head()

Dataset listo con 45754 filas.


,image_id,power_loss
0,solar_Fri_Jun_16_10__0__11_2017_L_0.9061532083...,0.906153
1,solar_Fri_Jun_16_10__0__16_2017_L_0.9030816970...,0.903082
2,solar_Fri_Jun_16_10__0__1_2017_L_0.91669804403...,0.916698
3,solar_Fri_Jun_16_10__0__21_2017_L_0.9030816970...,0.903082
4,solar_Fri_Jun_16_10__0__26_2017_L_0.8960873911...,0.896087
